# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 05: Pydantic v2 / DTOs en PHP y Tipado Estático
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *DTO manual con validación de tipos primitivos*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Esquema Pydantic v2 con validadores personalizados* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de tipos incompatibles y coerción*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion05.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** DTO manual con validación de tipos primitivos

In [4]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano - Sesión 05")

# DTO parA POR DER HACER LA PRUEBA
class PayloadRequest(BaseModel):
    dato: str

# DPARA EL RETO
class ProductoManual(BaseModel):
    nombre: str
    cantidad: str
    precio: str

@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):

    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

@app.post("/api/producto/manual")
def crear_producto_manual(prod: ProductoManual):
    # ValidaR manual de tipos primitivos
    if not prod.cantidad.isdigit():
        raise HTTPException(status_code=400, detail="La cantidad debe ser un entero")
    if not prod.precio.replace('.', '', 1).isdigit():
        raise HTTPException(status_code=400, detail="El precio debe ser un número")
    if int(prod.cantidad) <= 0 or float(prod.precio) <= 0:
        raise HTTPException(status_code=400, detail="Los valores deben ser mayores a 0")

    return {
        "status": "success",
        "mensaje": "Validación manual exitosa",
        "datos": {"nombre": prod.nombre, "cantidad": int(prod.cantidad), "precio": float(prod.precio)}
    }

Overwriting app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [5]:
# Pega aquí el código generado por la IA para auditarlo
%%writefile app_ia.py
from fastapi import FastAPI
from pydantic import BaseModel, Field, field_validator, StrictInt

app = FastAPI(title="API Científico - Sesión 05")

# DTO para la prueba del profesor
class PayloadRequest(BaseModel):
    dato: str

# DTO Optimizado con Pydantic v2
class ProductoOptimizado(BaseModel):
    nombre: str = Field(..., min_length=3)
    cantidad: StrictInt = Field(..., gt=0)
    precio: float = Field(..., gt=0)

    @field_validator('nombre')
    @classmethod
    def nombre_sin_espacios(cls, v: str) -> str:
        if not v.strip():
            raise ValueError('El nombre no puede estar vacío')
        return v.strip().title()

@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

@app.post("/api/producto/optimizado")
def crear_producto_optimizado(prod: ProductoOptimizado):
    return {
        "status": "success",
        "mensaje": "Validado por Pydantic v2",
        "producto": prod.nombre,
        "cantidad": prod.cantidad
    }


Writing app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de tipos incompatibles y coerción

In [3]:
import threading
import time
import uvicorn
import httpx
from app_artesano import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 05 ===\n")
r = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"Respuesta Servidor: HTTP {r.status_code} - {r.json()}")
assert r.status_code == 200
print("PASO ✅ - Test Harness HTTP ejecutado con exito.")


=== AUDITORIA CRITICA WEB: SESION 05 ===

Respuesta Servidor: HTTP 200 - {'status': 'success', 'mensaje': 'Procesado: Prueba Test Harness'}
PASO ✅ - Test Harness HTTP ejecutado con exito.
